# Project 1 

## Author: Yefrid Cordoba

## ST - 554 Big data analysis

## Part I

In this part, a class`SparkDataCheck` was created to perform a data check for the airquality data from the UC Irvine ML repo, including different ways to create instances for the class, checking that the some numeric variable falls in a user defined range, or showing frequencies for 1 or 2 non-numeric variables.\
The development of this class is important to add to our tools that we are going to use in the end of the course for data that is obtained overtime.

First we initialize the Spark session

In [242]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

Then, the module that contains the class is imported into the session

In [243]:
import Project2_class_YC
import pandas as pd

In [244]:
import importlib
importlib.reload(Project2_class_YC)

<module 'Project2_class_YC' from '/home/jupyter-yacordob@ncsu.edu/Project2_class_YC.py'>

Now, an instance for the class is created from the csv file, which is saved ina folder called `Data`, relative to my working directory.

In [245]:
air_data = Project2_class_YC.SparkDataCheck.createfrom_csv(spark, 'Data/air.csv')

It is necessary to rename the columns that have notation that can be undertand in a different way by the program (e.g., `'PT08.S1(CO)'` to `'S1(CO)'` )

In [246]:
air_data.df = air_data.df.withColumnsRenamed({\
                                              'PT08.S1(CO)':'S1(CO)', 'PT08.S2(NMHC)':'S2(NMHC)', 'PT08.S3(NOx)':'S3(NOx)', \
                                              'PT08.S4(NO2)': 'S4(NO2)', 'PT08.S5(O3)':'S5(O3)'})

In [247]:
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+
|  0|3/10/2004|2026-03-24 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-24 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-24 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-24 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.7867|
|  4|3/10/2004|2026-03-24 22:00:00|   1.6|  1272|      51|     6.5|  

26/03/24 19:27:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


Showinf the first 10 rows of the instance of the class created, from which the last row (index 9), contains missing values, which are shown as `-200` in the dataframe./
For that reason, it is necessary to replace those values in the dataframe using the method `df.replace()` to create the Null values recognizable by pySpark.

In [248]:
air_data.df = air_data.df.replace(-200, None)

Checking that the `-200` changed for `Null` (or None)in the dataframe.

In [249]:
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+
|  0|3/10/2004|2026-03-24 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-24 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-24 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-24 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.7867|
|  4|3/10/2004|2026-03-24 22:00:00|   1.6|  1272|      51|     6.5|  

26/03/24 19:27:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


### Examples of the class methods

#### When all bounds are included for a numeric variable

First we are going to check when it is numeric and both of the limits are included in the method ()which should work whitout exeptions

In [250]:
air_data.create_range_boolean('NO2(GT)', lower = 90, upper = 115)

DataFrame[_c0: int, Date: string, Time: timestamp, CO(GT): double, S1(CO): int, NMHC(GT): int, C6H6(GT): double, S2(NMHC): int, NOx(GT): int, S3(NOx): int, NO2(GT): int, S4(NO2): int, S5(O3): int, T: double, RH: double, AH: double, Is_in_range: boolean]

In [251]:
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|Is_in_range|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|  0|3/10/2004|2026-03-24 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|       true|
|  1|3/10/2004|2026-03-24 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|       true|
|  2|3/10/2004|2026-03-24 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|       true|
|  3|3/10/2004|2026-03-24 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.786

26/03/24 19:27:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


It is clear that based on the range provided it included what it was supposed to, also the Null values were included as Null in the new column created.

#### When lower limit is missing

In this part we are going to check the code when lower bound is missing.

In [252]:
air_data.create_range_boolean('NO2(GT)', upper = 115)
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|Is_in_range|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|  0|3/10/2004|2026-03-24 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|       true|
|  1|3/10/2004|2026-03-24 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|       true|
|  2|3/10/2004|2026-03-24 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|       true|
|  3|3/10/2004|2026-03-24 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.786

26/03/24 19:27:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


Of course the values that were below the lower limit previously now are included in the range.

#### When upper limit is missing

Now the upper limit is not going to be included in the method.

In [253]:
air_data.create_range_boolean('NO2(GT)', lower = 90)
air_data.df.show(10)

+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|_c0|     Date|               Time|CO(GT)|S1(CO)|NMHC(GT)|C6H6(GT)|S2(NMHC)|NOx(GT)|S3(NOx)|NO2(GT)|S4(NO2)|S5(O3)|   T|  RH|    AH|Is_in_range|
+---+---------+-------------------+------+------+--------+--------+--------+-------+-------+-------+-------+------+----+----+------+-----------+
|  0|3/10/2004|2026-03-24 18:00:00|   2.6|  1360|     150|    11.9|    1046|    166|   1056|    113|   1692|  1268|13.6|48.9|0.7578|       true|
|  1|3/10/2004|2026-03-24 19:00:00|   2.0|  1292|     112|     9.4|     955|    103|   1174|     92|   1559|   972|13.3|47.7|0.7255|       true|
|  2|3/10/2004|2026-03-24 20:00:00|   2.2|  1402|      88|     9.0|     939|    131|   1140|    114|   1555|  1074|11.9|54.0|0.7502|       true|
|  3|3/10/2004|2026-03-24 21:00:00|   2.2|  1376|      80|     9.2|     948|    172|   1092|    122|   1584|  1203|11.0|60.0|0.786

26/03/24 19:27:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-yacordob@ncsu.edu/Data/air.csv


And all the values that are above the lower limit are included now.

#### Non-numeric column is added

The column date now is going to be the input for our method to test the printed message

In [254]:
air_data.create_range_boolean('Date', lower = 90)


The column Date is not a numeric column


DataFrame[_c0: int, Date: string, Time: timestamp, CO(GT): double, S1(CO): int, NMHC(GT): int, C6H6(GT): double, S2(NMHC): int, NOx(GT): int, S3(NOx): int, NO2(GT): int, S4(NO2): int, S5(O3): int, T: double, RH: double, AH: double, Is_in_range: boolean]

We see that there is no modification to the dataframe and the message is raised because the column `Date` is not numric type.

#### Checking summarization methods

First using both arguments for the summarization method.

In [255]:
air_data.get_min_max(column = 'CO(GT)', group = 'Date')

,Date,max_CO(GT),min_CO(GT)
0,9/2/2004,5.2,0.6
1,12/26/2004,3.8,0.3
2,2/18/2005,4.8,0.4
3,10/10/2004,3.7,0.7
4,10/11/2004,4.8,0.4
...,...,...,...
386,1/23/2005,0.8,0.8
387,6/28/2004,5.6,0.4
388,8/16/2004,2.1,0.3
389,12/20/2004,4.1,0.6


When just one of the arguments is given

In [256]:
air_data.get_min_max(column = 'S2(NMHC)')

,max_S2(NMHC),min_S2(NMHC)
0,2214,383


Now trying to rise an error

In [257]:
air_data.get_min_max(column = 'Date')

The column Date is not a numeric column


It is seen that the code works how is intended to.

### Reading as a `pd.DataFrame`

We are going to imort the data now as a pandas dataframe, and from that dataframe we are going to create and instance of the class using the `@classmethod` created in the class

In [258]:
pd_air_data = pd.read_csv('Data/air.csv')

In [259]:
f_air_data = Project2_class_YC.SparkDataCheck.createfrom_pandas(spark, pd_air_data)

#### Example for the methods

We are going to check the count for the variable `Date` in the dataframe

In [260]:
f_air_data.get_counts('Date')

,Date,Count
0,3/11/2004,24
1,3/12/2004,24
2,3/10/2004,6
3,3/13/2004,24
4,3/15/2004,24
...,...,...
386,3/30/2005,24
387,3/31/2005,24
388,4/3/2005,24
389,4/4/2005,15


This table shows what is the number of measurements per day during the time the digital nose was recording air quality data from the Italian city.

## Part II

In this section, the weekly NFL data is going to be filtered and some summary statistic are going to be calculated using the API pandas on spark, and spark SQL. The objective is to get a deep understanding and practice on the most common and useful tools when dealing with data, including filtering, sorting, cleaning, and manipulating different datasets.

Importing the modules that are necessary.

In [261]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
import pandas as pd
import pyspark.pandas as ps

spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

Now we create a pandas on spark dataframe with the nfl data.

In [262]:
nfl_data = ps.read_csv('Data/weekly_nfl_data.csv', sep = ',')

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


Checking the first 5 rows of the 

In [263]:
nfl_data.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


Now we are going to extract all the column names and the info related to each column.

In [264]:
nfl_data.info()

<class 'pyspark.pandas.frame.DataFrame'>
Index: 128873 entries, 0 to 128872
Data columns (total 53 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   player_id                    128873 non-null  object 
 1   player_name                  61493 non-null   object 
 2   player_display_name          128870 non-null  object 
 3   position                     128801 non-null  object 
 4   position_group               128801 non-null  object 
 5   headshot_url                 69819 non-null   object 
 6   recent_team                  128873 non-null  object 
 7   season                       128873 non-null  int32  
 8   week                         128873 non-null  int32  
 9   season_type                  128873 non-null  object 
 10  opponent_team                128873 non-null  object 
 11  completions                  128873 non-null  int32  
 12  attempts                     128873 non-null  int32  
 13  p

26/03/24 19:27:42 WARN AttachDistributedSequenceExec: clean up cached RDD(1409) in AttachDistributedSequenceExec(7351)


### Stats for QB in regular season 2003 to 2023

We are going to take the original data and perform filtering for quarterback `'QB'`, during the regular season between the 2005 and 2023, for combinations of the name of the players and the season (using the `.groupBy()` method), and finally calculating the sum and the mean for eachof the combiantions.

In [265]:
# On the first line the data is being filtered across different columns with the values especified.
# On the next line we select just the columns that we are going to keep after the data has been filteres
# On the next line it has been grouped by two categorical variables that are the name and the year
# On the las line, the function will calculate sum and mean for all the previously grouped variables.
QB_reg = nfl_data.loc[((nfl_data.position == 'QB') & (nfl_data.season_type == 'REG') & (nfl_data.season >= 2005) & (nfl_data.season <= 2023))\
                      ,['player_display_name', 'season', 'week', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions'] ]\
                    .groupby(['player_display_name', 'season'])\
                    .agg(['sum', 'mean'])
                    

In [266]:
QB_reg.info()

<class 'pyspark.pandas.frame.DataFrame'>
MultiIndex: 1456 entries, (Jake Delhomme, 2006) to (Will Levis, 2023)
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   (week, sum)            1456 non-null   int64  
 1   (week, mean)           1456 non-null   float64
 2   (completions, sum)     1456 non-null   int64  
 3   (completions, mean)    1456 non-null   float64
 4   (attempts, sum)        1456 non-null   int64  
 5   (attempts, mean)       1456 non-null   float64
 6   (passing_yards, sum)   1456 non-null   float64
 7   (passing_yards, mean)  1456 non-null   float64
 8   (passing_tds, sum)     1456 non-null   int64  
 9   (passing_tds, mean)    1456 non-null   float64
 10  (interceptions, sum)   1456 non-null   float64
 11  (interceptions, mean)  1456 non-null   float64
dtypes: float64(8), int64(4)

In this code chunk two new variables are included where the completion percentage and the ratio of passing to interceptions are calculated and included in a new `ps.DataFrame`

In [267]:
#This line is claculating the completion percentage variable and adding it as a new column to the pandas dataframe.
QB_reg['completion_percentage'] = QB_reg[('completions'), ('sum')] / QB_reg[('attempts'), ('sum')]

#This line is claculating the ratio variable and adding it as a new column to the pandas dataframe.
QB_reg['td_int_ratio'] = QB_reg[('passing_tds'), ('sum')] / QB_reg[('interceptions'), ('sum')]

In [268]:
QB_reg.head()

week            completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                                      
Jake Delhomme       2006     99   7.615385         263  20.230769      431  33.153846        2805.0  215.769231          17  1.307692          11.0  0.846154              0.610209     1.545455
Jake Plummer        2005    144   9.000000         277  17.312500      456  28.500000        3366.0  210.375000          18  1.125000           7.0  0.437500              0.607456     2.571429
Matt Schaub         2006     60  12.000000          18   3.600000       27   5.400000         208.0   41.600000           1  0.200000           2.0  0.400000              0.666667     0.500000
Vince Young         2006    143   9.533333         184  12.266667      356  23.733333        2199.0  146.600000          12  0.800000          13.0  0.866667              0.516854     0.923077
Kerry Collins       2007     48   8.000000          50   8.333333       82  13.666667         531.0   88.500000           0  0.000000           0.0  0.000000              0.609756          NaN

### Subsetting the new dataframe

First we are going to filter the values when the sum of attempts is at least 50, and sorted by the completion percentage, showing jus the first 40 rows of the dataframe.

In [269]:
#First it is necessary to filter the data and then a descending sorting is performed based on the completion percentage.
QB_reg.loc[QB_reg[('attempts', 'sum')] >= 50]\
        .sort_values(by = 'completion_percentage', ascending = False).head(40)

week            completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                                      
C.J. Beathard       2023     65  10.833333          40   6.666667       53   8.833333         349.0   58.166667           1  0.166667           0.0  0.000000              0.754717          inf
Colt McCoy          2021     62   8.857143          74  10.571429       99  14.142857         740.0  105.714286           3  0.428571           1.0  0.142857              0.747475     3.000000
Matt Schaub         2019     52  10.400000          50  10.000000       67  13.400000         580.0  116.000000           3  0.600000           1.0  0.200000              0.746269     3.000000
Drew Brees          2018    130   8.666667         364  24.266667      489  32.600000        3992.0  266.133333          32  2.133333           5.0  0.333333              0.744376     6.400000
                    2019    119  10.818182         281  25.545455      378  34.363636        2979.0  270.818182          27  2.454545           4.0  0.363636              0.743386     6.750000
Mason Rudolph       2023     66  16.500000          55  13.750000       74  18.500000         719.0  179.750000           3  0.750000           0.0  0.000000              0.743243          inf
Taysom Hill         2020    147   9.187500          88   5.500000      121   7.562500         928.0   58.000000           4  0.250000           2.0  0.125000              0.727273     2.000000
Nick Foles          2018     51  10.200000         141  28.200000      195  39.000000        1413.0  282.600000           7  1.400000           4.0  0.800000              0.723077     1.750000
Drew Brees          2017    148   9.250000         386  24.125000      536  33.500000        4334.0  270.875000          23  1.437500           8.0  0.500000              0.720149     2.875000
Sam Bradford        2016    146   9.733333         395  26.333333      552  36.800000        3877.0  258.466667          20  1.333333           5.0  0.333333              0.715580     4.000000
Drew Brees          2011    142   8.875000         471  29.437500      660  41.250000        5535.0  345.937500          46  2.875000          14.0  0.875000              0.713636     3.285714
Colt McCoy          2014     57  11.400000          91  18.200000      128  25.600000        1057.0  211.400000           4  0.800000           3.0  0.600000              0.710938     1.333333
Aaron Rodgers       2020    148   9.250000         372  23.250000      526  32.875000        4299.0  268.687500          48  3.000000           5.0  0.312500              0.707224     9.600000
Bailey Zappe        2022     22   5.500000          65  16.250000       92  23.000000         781.0  195.250000           5  1.250000           3.0  0.750000              0.706522     1.666667
Drew Brees          2009    131   8.733333         363  24.200000      514  34.266667        4388.0  292.533333          34  2.266667          11.0  0.733333              0.706226     3.090909
                    2020     97   8.083333         275  22.916667      390  32.500000        2942.0  245.166667          24  2.000000           6.0  0.500000              0.705128     4.000000
Joe Burrow          2021    143   8.937500         366  22.875000      520  32.500000        4611.0  288.187500          34  2.125000          14.0  0.875000              0.703846     2.428571
Derek Carr          2019    147   9.187500         361  22.562500      513  32.062500        4054.0  253.375000          21  1.312500           8.0  0.500000          

Now we are going to get the values sorted by the `td_int_ratio` and show just the first 40 rows.

In [270]:
#Now the same filtering, but with the sorting based on the ratio variable.
QB_reg.loc[QB_reg[('attempts', 'sum')] >= 50]\
        .sort_values(by = 'td_int_ratio', ascending = False).head(40)

week            completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                                      
Charlie Batch       2006     71  10.142857          30   4.285714       52   7.428571         477.0   68.142857           5  0.714286           0.0  0.000000              0.576923          inf
Matt Schaub         2005     65   9.285714          33   4.714286       64   9.142857         495.0   70.714286           4  0.571429           0.0  0.000000              0.515625          inf
Todd Collins        2007     62  15.500000          67  16.750000      105  26.250000         888.0  222.000000           5  1.250000           0.0  0.000000              0.638095          inf
Troy Smith          2007     62  15.500000          40  10.000000       76  19.000000         452.0  113.000000           2  0.500000           0.0  0.000000              0.526316          inf
Jake Locker         2011     51  10.200000          34   6.800000       66  13.200000         542.0  108.400000           4  0.800000           0.0  0.000000              0.515152          inf
Brian Hoyer         2016     27   4.500000         134  22.333333      200  33.333333        1445.0  240.833333           6  1.000000           0.0  0.000000              0.670000          inf
Nick Foles          2016     17   8.500000          36  18.000000       55  27.500000         410.0  205.000000           3  1.500000           0.0  0.000000              0.654545          inf
Derek Anderson      2014     30   6.000000          65  13.000000       97  19.400000         701.0  140.200000           5  1.000000           0.0  0.000000              0.670103          inf
Jimmy Garoppolo     2016     49   8.166667          43   7.166667       63  10.500000         502.0   83.666667           4  0.666667           0.0  0.000000              0.682540          inf
Matt Moore          2019     54   9.000000          59   9.833333       91  15.166667         659.0  109.833333           4  0.666667           0.0  0.000000              0.648352          inf
C.J. Beathard       2020     67  11.166667          66  11.000000      104  17.333333         787.0  131.166667           6  1.000000           0.0  0.000000              0.634615          inf
Andy Dalton         2023      8   4.000000          34  17.000000       58  29.000000         361.0  180.500000           2  1.000000           0.0  0.000000              0.586207          inf
Mason Rudolph       2023     66  16.500000          55  13.750000       74  18.500000         719.0  179.750000           3  0.750000           0.0  0.000000              0.743243          inf
Desmond Ridder      2022     66  16.500000          73  18.250000      115  28.750000         708.0  177.000000           2  0.500000           0.0  0.000000              0.634783          inf
C.J. Beathard       2023     65  10.833333          40   6.666667       53   8.833333         349.0   58.166667           1  0.166667           0.0  0.000000              0.754717          inf
Tom Brady           2016    134  11.166667         291  24.250000      432  36.000000        3554.0  296.166667          28  2.333333           2.0  0.166667              0.673611    14.000000
Nick Foles          2013    129   9.923077         203  15.615385      317  24.384615        2891.0  222.384615          27  2.076923           2.0  0.153846              0.640379    13.500000
Josh McCown         2013     92  11.500000         149  18.625000      224  28.000000        1829.0  228.625000          13  1.625000           1.0  0.125000          

It can be seen that as some of the sum of interceptions is 0, the first set of vlaues for the ratio yields the value `inf`.

### The same now with `Spark SQL DataFrame`

We start by importing the data as a Spark SQL DataFrame.

In [271]:
#We import the data as a spark SQL DataFrame
nfld_spark = spark.read.load('Data/weekly_nfl_data.csv',
                            format = 'csv',
                            sep = ',',
                            inferSchema = 'true',
                            header = 'true')

Then we get the first 5 rows using the method `.show()` on the dataframe,.

In [272]:
nfld_spark.show(3)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

Then we report all the column names.

In [273]:
nfld_spark.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

#### Subsetting on Spark SQL

We are going to take the original data and perform filtering for quarterback `'QB'`, during the regular season between the 2005 and 2023, for combinations of the name of the players and the season (using the `.groupBy()` method), and finally calculating the sum and the mean for each of the combiantions.

In [274]:
# On the first line, we select just the columns that we are going to keep after the data has been filteres
# On the next three lines, the data is being filtered across different columns with the values especified.
# On the las line, the data is being grouped.
QB_reg2 = nfld_spark.select(['player_display_name', 'season', 'week', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions'])\
                    .where(nfld_spark.position == 'QB')\
                    .where(nfld_spark.season_type == 'REG')\
                    .where((nfld_spark.season >=2005) & (nfld_spark.season <=2023))\
                    .groupBy(['player_display_name', 'season'])

After a grouped object has been generated to make this process easy, two tables are going to be claculated and join in the column of the player name, dropping the columns that do not make sense to calculate statistics (`week` and `season`) 

In [275]:
QB_reg2.sum().show(10)

+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+
|player_display_name|season|sum(season)|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|
+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+
|      Jake Delhomme|  2006|      26078|       99|             263|          431|            2805.0|              17|              11.0|
|       Jake Plummer|  2005|      32080|      144|             277|          456|            3366.0|              18|               7.0|
|        Matt Schaub|  2006|      10030|       60|              18|           27|             208.0|               1|               2.0|
|        Vince Young|  2006|      30090|      143|             184|          356|            2199.0|              12|              13.0|
|      Kerry Collins|  2007|      12042| 

In [276]:
#The result of aggregating the variables by sum is going to be join with the result of aggregating the variables by mean on the two variables of interest.
QB_reg2 = QB_reg2.sum()\
                 .join(QB_reg2.mean(), on = ['player_display_name', 'season'])\
                 .drop('avg(week)', 'avg(season)', 'sum(week)', 'sum(season)') 
#Four variables that do not make sense to calculate sum and mean are dropped.
QB_reg2.show(3)

+-------------------+------+----------------+-------------+------------------+----------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|player_display_name|season|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)| avg(completions)|    avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|
+-------------------+------+----------------+-------------+------------------+----------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|      Jake Delhomme|  2006|             263|          431|            2805.0|              17|              11.0|20.23076923076923|33.15384615384615|215.76923076923077|1.3076923076923077|0.8461538461538461|
|       Jake Plummer|  2005|             277|          456|            3366.0|              18|               7.0|          17.3125|             28.5|           210.375

Now we are going to calculate some other parameters that important to know the performance of the players.

In [277]:
#Creating the two new variables of performance for each of the players per year, passing them as a dictionary.
QB_reg2 = QB_reg2.withColumns({'completion_percentage': QB_reg2['sum(completions)'] / QB_reg2['sum(attempts)'], \
                               'td_int_ratio': QB_reg2['sum(passing_tds)'] / QB_reg2['sum(interceptions)']})

In [278]:
QB_reg2.show(3)

+-------------------+------+----------------+-------------+------------------+----------------+------------------+-----------------+-----------------+------------------+------------------+------------------+---------------------+------------------+
|player_display_name|season|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)| avg(completions)|    avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|completion_percentage|      td_int_ratio|
+-------------------+------+----------------+-------------+------------------+----------------+------------------+-----------------+-----------------+------------------+------------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|             263|          431|            2805.0|              17|              11.0|20.23076923076923|33.15384615384615|215.76923076923077|1.3076923076923077|0.8461538461538461|   0.6102088167053364|1.5454545454545454|
|   

After the columns have been created now we filter the values where the sum of attempts is larger or equal to 50, then the values are going to be sorted based on completion percentage.

In [279]:
#First the new dataset is being filtered for the players with a larger or equal to 50 total attempts.
QB_reg2 = QB_reg2.filter(QB_reg2['sum(attempts)'] >= 50)


Showing the first 40 rows

In [280]:
#The data has been sorted descending and to get a better viasualization of the data, the data set is being converted to a Pandas Dataframe
QB_reg2.sort(QB_reg2['completion_percentage'], ascending = False).toPandas().head(40)

,player_display_name,season,sum(completions),sum(attempts),sum(passing_yards),sum(passing_tds),sum(interceptions),avg(completions),avg(attempts),avg(passing_yards),avg(passing_tds),avg(interceptions),completion_percentage,td_int_ratio
0,C.J. Beathard,2023,40,53,349.0,1,0.0,6.666667,8.833333,58.166667,0.166667,0.000000,0.754717,NaN
1,Colt McCoy,2021,74,99,740.0,3,1.0,10.571429,14.142857,105.714286,0.428571,0.142857,0.747475,3.000000
2,Matt Schaub,2019,50,67,580.0,3,1.0,10.000000,13.400000,116.000000,0.600000,0.200000,0.746269,3.000000
3,Drew Brees,2018,364,489,3992.0,32,5.0,24.266667,32.600000,266.133333,2.133333,0.333333,0.744376,6.400000
4,Drew Brees,2019,281,378,2979.0,27,4.0,25.545455,34.363636,270.818182,2.454545,0.363636,0.743386,6.750000
5,Mason Rudolph,2023,55,74,719.0,3,0.0,13.750000,18.500000,179.750000,0.750000,0.000000,0.743243,NaN
6,Taysom Hill,2020,88,121,928.0,4,2.0,5.500000,7.562500,58.000000,0.250000,0.125000,0.727273,2.000000
7,Nick Foles,2018,141,195,1413.0,7,4.0,28.200000,39.000000,282.600000,1.400000,0.800000,0.723077,1.750000
8,Drew Brees,2017,386,536,4334.0,23,8.0,24.125000,33.500000,270.875000,1.437500,0.500000,0.720149,2.875000
9,Sam Bradford,2016,395,552,3877.0,20,5.0,26.333333,36.800000,258.466667,1.333333,0.333333,0.715580,4.000000


Now we sort by the `td_int_ratio`

In [281]:
#The process is the same, just sorting by a different variable.
QB_reg2.sort(QB_reg2['td_int_ratio'], ascending = False).toPandas().head(40)

,player_display_name,season,sum(completions),sum(attempts),sum(passing_yards),sum(passing_tds),sum(interceptions),avg(completions),avg(attempts),avg(passing_yards),avg(passing_tds),avg(interceptions),completion_percentage,td_int_ratio
0,Tom Brady,2016,291,432,3554.0,28,2.0,24.250000,36.000000,296.166667,2.333333,0.166667,0.673611,14.000000
1,Nick Foles,2013,203,317,2891.0,27,2.0,15.615385,24.384615,222.384615,2.076923,0.153846,0.640379,13.500000
2,Josh McCown,2013,149,224,1829.0,13,1.0,18.625000,28.000000,228.625000,1.625000,0.125000,0.665179,13.000000
3,Aaron Rodgers,2018,372,597,4442.0,25,2.0,23.250000,37.312500,277.625000,1.562500,0.125000,0.623116,12.500000
4,Damon Huard,2006,148,244,1878.0,11,1.0,14.800000,24.400000,187.800000,1.100000,0.100000,0.606557,11.000000
5,Aaron Rodgers,2020,372,526,4299.0,48,5.0,23.250000,32.875000,268.687500,3.000000,0.312500,0.707224,9.600000
6,Aaron Rodgers,2021,366,531,4115.0,37,4.0,22.875000,33.187500,257.187500,2.312500,0.250000,0.689266,9.250000
7,Tom Brady,2010,324,492,3900.0,36,4.0,20.250000,30.750000,243.750000,2.250000,0.250000,0.658537,9.000000
8,Jake Delhomme,2007,55,86,617.0,8,1.0,18.333333,28.666667,205.666667,2.666667,0.333333,0.639535,8.000000
9,Aaron Rodgers,2014,341,520,4381.0,38,5.0,21.312500,32.500000,273.812500,2.375000,0.312500,0.655769,7.600000


It is noted that the names for the table where the ratio of passing to interceptions does not match with the one generated only through pandas-on-spark, and this is because spark SQL handle infinite numbers as null values, while pandas handle those values as a inf, which are generated when a number is divided by 0.